# MT enumeradora de palíndromos binarios (3 cintas)


**NOTA:** ejecuta las celdas en orden (Entorno ▸ Ejecutar todo).

| Sección | Qué contiene |
|---|---|
| 1 | Constantes y la clase `Cinta` |
| 2 | Simulador `MTEnumeradora` (aplica δ a las tres cintas) |
| 3 | Función de transición δ: bloques `G_*`, `C_*`, `D_*`, `P_*` |
| 4 | Validación exhaustiva |
| 5 | Visualización interactiva |
| 6 | Demos |


## 1. Constantes y la clase `Cinta`
**Constantes**
- `B = 'λ'`: símbolo blanco.
- `SIGMA = {0,1}`: alfabeto de entrada Σ.
- `GAMMA = {λ,0,1,X,#}`: alfabeto de cinta Γ. `X` es la marca del decisor y `#` el separador de la salida.
- `MOV`: traduce `L`, `S`, `R` a −1, 0, +1 (izquierda, quedarse, derecha).

**`Cinta`** modela una cinta infinita hacia la derecha (crece con blancos). La celda 0 siempre vale `λ` y funciona como **tope izquierdo**:
si un cabezal intenta salirse por la izquierda se lanza un error, porque significaría que un rebobinado falló.

In [1]:
from itertools import product

B = 'λ'                              # símbolo blanco
SIGMA = ['0', '1']                   # alfabeto de entrada Σ
GAMMA = [B, '0', '1', 'X', '#']      # alfabeto de cinta Γ (X = marca del decisor, # = separador de salida)
MOV = {'L': -1, 'S': 0, 'R': +1}     # L = izquierda, S = quedarse, R = derecha


class Cinta:
    """Cinta infinita a la derecha; la celda 0 siempre es λ (tope izquierdo)."""
    def __init__(self, contenido, cab=0):
        self.c = list(contenido)
        self.cab = cab
        while self.cab >= len(self.c):
            self.c.append(B)

    def leer(self):
        return self.c[self.cab]

    def escribir(self, s):
        self.c[self.cab] = s

    def mover(self, m):
        self.cab += MOV[m]
        # Si un cabezal cruza la celda 0, el rebobinado falló: error de diseño, no comportamiento válido.
        if self.cab < 0:
            raise RuntimeError("Error de diseño: el cabezal se salió por la izquierda")
        while self.cab >= len(self.c):      # la cinta crece a la derecha con blancos
            self.c.append(B)

## 2. Simulador `MTEnumeradora`
Una **sola** función de transición δ gobierna las tres cintas a la vez (MT multicinta determinista).

**Cintas al iniciar**
| Cinta | Rol | Contenido inicial |
|---|---|---|
| 1 | Generador | `λ` (candidata ε) |
| 2 | Decisor | `λ` |
| 3 | Impresora | `λ`, **vacía**, con el cabezal en la primera celda libre |

**Métodos**
- `paso()`: lee las tres celdas bajo los cabezales, busca la transición, escribe y mueve las tres cintas. Si no existe la transición lanza error (δ es parcial a propósito).
- `foto()`: guarda en cada paso el estado, el contenido de las 3 cintas y la posición de los 3 cabezales. Este log es lo que lee la visualización.
- `ejecutar(n_candidatas=…, n_aceptadas=…)`: repite `paso()` hasta procesar *n* candidatas o imprimir *n* cadenas.
- `salida()`: devuelve el contenido de la cinta 3.

**Registros de apoyo:** `veredictos` (candidata, decisión y pasos del decisor), `salidas` (cinta 3 tras cada impresión) e `invariantes` (posición de cabezales al iniciar cada ciclo).

**Verificaciones externas.** `verificar_cota` y `max_pasos` **no forman parte de la MT**: solo comprueban desde fuera que el decisor termina dentro de $O(n^2)$ y evitan que una prueba se cuelgue por accidente. δ decide por sí sola.

In [2]:
class MTEnumeradora:
    """MT determinista de 3 cintas: cinta 1 = Generador, cinta 2 = Decisor, cinta 3 = Impresora.
        δ: Q × Γ³ → Q × Γ³ × {L,S,R}³ gobierna las tres cintas a la vez."""
    def __init__(self, delta, candidata_inicial=''):
        self.delta = delta
        self.cintas = [Cinta([B] + list(candidata_inicial)),   # cinta 1: candidata actual (ε = cinta vacía)
                       Cinta([B]),                             # cinta 2: espacio de trabajo del decisor
                       Cinta([B], cab=1)]                      # cinta 3: VACÍA; cabezal en la 1ª celda libre
        self.estado = 'G_Inicio'      # el generador produce ε como primera candidata
        self.log = []                 # una foto completa de la MT por paso (estado + 3 cintas + 3 cabezales)
        self.inicios = []             # (índice en el log, candidata) al comenzar cada ciclo
        self.veredictos = []          # (candidata, 'Acepta'/'Rechaza', pasos del decisor)
        self.salidas = []             # contenido de la cinta 3 tras cada impresión
        self.invariantes = []         # (cab1, cab2, cab3, len(salida)) al iniciar cada ciclo

    def foto(self):
        """Registro del paso: estado, contenido y cabezal de cada una de las 3 cintas."""
        return (self.estado, [list(t.c) for t in self.cintas], [t.cab for t in self.cintas])

    def candidata(self):
        return ''.join(s for s in self.cintas[0].c if s in '01')

    def paso(self):
        """Un paso de la MT: lee las 3 celdas, busca δ, escribe y mueve las 3 cintas."""
        # Configuración (q, a1, a2, a3): las tres cabezas se leen simultaneamente;
        # δ(q, a1, a2, a3) = (q', b1, b2, b3, m1, m2, m3) se aplica de forma atómica a las tres cintas.
        lecturas = tuple(t.leer() for t in self.cintas)
        clave = (self.estado, lecturas)
        # Evita que un fallo de sincronización pase inadvertido.
        if clave not in self.delta:
            raise RuntimeError(f"Transición indefinida: {clave}")
        q, esc, mov = self.delta[clave]
        for t, e, m in zip(self.cintas, esc, mov):
            t.escribir(e)
            t.mover(m)
        self.estado = q

    def ejecutar(self, n_candidatas=None, n_aceptadas=None, guardar_log=True,
                 verificar_cota=True, max_pasos=None):
        """Corre el ciclo hasta procesar n_candidatas candidatas o imprimir n_aceptadas cadenas.
        NOTA: la MT no depende de ningún límite para decidir; `verificar_cota` y `max_pasos`
        son verificaciones externas de seguridad del simulador, ajenas a δ."""
        if n_candidatas is None and n_aceptadas is None:
            raise ValueError("Indica n_candidatas o n_aceptadas")
        if guardar_log:
            self.log = [self.foto()]
        self.inicios = [(0, self.candidata())]
        terminadas = aceptadas = pasos_dec = pasos_total = 0
        cand_dec = ''
        while True:
            previo = self.estado
            self.paso()
            pasos_total += 1
            if max_pasos is not None and pasos_total > max_pasos:      # verificacion externa
                raise RuntimeError("Simulación detenida por el límite externo de seguridad")
            if guardar_log:
                self.log.append(self.foto())
            if self.estado == 'D_0':
                pasos_dec, cand_dec = 0, self.candidata()
            if self.estado.startswith('D_'):
                pasos_dec += 1
            if self.estado in ('D_Acepta', 'D_Rechaza'):
                if verificar_cota:
                    # Terminación: cada ronda marca ≥ 1 símbolo nuevo, así que el potencial
                    # "símbolos sin marcar" decrece -> ≤ ⌈n/2⌉+1 rondas, cada una de costo ≤ 2(n+2)
                    # (ida y vuelta) ⇒ T_D(n) = O(n²). Este no participa en δ: solo comprueba
                    # La cota (4(n+2)²) y prueba que D es un decisor genuino, no un mero reconocedor.
                    assert pasos_dec <= 4 * (len(cand_dec) + 2) ** 2, "El decisor excedió la cota"
                self.veredictos.append((cand_dec, self.estado[2:], pasos_dec))
            if previo == 'P_R' and self.estado == 'G_0':               # terminó una impresión
                aceptadas += 1
                self.salidas.append(self.salida())
            if self.estado == 'C_0':                                   # empieza un ciclo nuevo
                s = self.salida()
                # La sincronización: validar() exige (0, 0, |salida|+1) en cada entrada a C_0.
                self.invariantes.append((self.cintas[0].cab, self.cintas[1].cab, self.cintas[2].cab, len(s)))
                if previo != 'G_Inicio':
                    terminadas += 1
                    if n_candidatas is not None and terminadas >= n_candidatas:
                        break
                    self.inicios.append((len(self.log) - 1 if guardar_log else 0, self.candidata()))
            if n_aceptadas is not None and aceptadas >= n_aceptadas:
                break
        return self

    def salida(self):
        return ''.join(s for s in self.cintas[2].c if s != B)

## 3. Función de transición δ
`construir_delta()` devuelve un diccionario `(estado, (símbolo1, símbolo2, símbolo3)) → (estado', (escribe1, escribe2, escribe3), (mueve1, mueve2, mueve3))`.

**Notación de `agregar(q, lee, q_sig, escribe, mueve)`**
- `lee`: tres cadenas con los símbolos admisibles en cada cinta (se expande el producto cartesiano).
- `escribe`: tres caracteres; `=` significa reescribir lo que se leyó.
- `mueve`: tres letras `L/S/R`, una por cinta.
- Un `assert` garantiza que sea **determinista**: cada configuración tiene a lo sumo una transición.

**Invariante de sincronización:** al entrar a `C_0`, el cabezal 1 y el cabezal 2 están en la celda 0 y el cabezal 3 está en la primera celda libre de la salida. Los estados `_R`, `D_RWA` y `D_RWR` existen para restablecerlo.

### Generador (cinta 1)
Calcula el **sucesor shortlex** como incremento en numeral biyectivo base 2: `u0→u1`, `u01ᵏ→u10ᵏ`, `1ⁿ→0ⁿ⁺¹`.
| Estado | Función |
|---|---|
| `G_Inicio` | La candidata inicial es ε (cinta 1 vacía); cede el control a `C_0` |
| `G_0` | Entra a la cadena desde la celda 0 |
| `G_1` | Recorre hasta el final de la candidata |
| `G_2` | Retrocede aplicando el acarreo: `1→0` y sigue; `0→1` termina; `λ` indica desbordamiento |
| `G_E` | Desbordamiento: recorre los ceros y agrega uno más (longitud n+1) |
| `G_R` | Rebobina la cinta 1 a la celda 0 y pasa a `C_0` |

###Transferencia (cinta 1 → cinta 2)
| Estado | Función |
|---|---|
| `C_0` | Con ambos cabezales en la celda 0, avanza los dos a la celda 1 |
| `C_1` | Copia símbolo a símbolo, sobrescribiendo restos de `X` de la ronda anterior |
| `C_R` | Rebobina las dos cintas en paralelo hasta leer (λ, λ) y cede el control a `D_0` |

###Decisor (cinta 2)
**Idea:** marcar con `X` el extremo izquierdo sin marcar, recordar su valor en el estado, compararlo con el extremo derecho sin marcar y repetir hacia el centro.
| Estado | Función |
|---|---|
| `D_0` | Entra a la cadena |
| `D_q0` | Marca el extremo izquierdo (`0` o `1`); si no queda nada sin marcar, acepta |
| `D_R0`, `D_R1` | Van al extremo derecho recordando el símbolo marcado |
| `D_L0`, `D_L1` | Comparan con el extremo derecho: igual ⇒ marcan y vuelven; distinto ⇒ rechazan; `X` ⇒ acepta |
| `D_V` | Vuelve al extremo izquierdo sin marcar |
| `D_Acepta`, `D_Rechaza` | Veredicto |
| `D_RWA`, `D_RWR` | Rebobinan la cinta 2 a la celda 0 y ceden el control: aceptada → `P_0`, rechazada → `G_0` |

*Ejemplo con `010`:* `010 → X10 → X1X → XXX` ⇒ acepta, porque el `1` central queda solo.

###Impresora (cinta 3)
Copia desde la **cinta 1** (la cinta 2 quedó con marcas `X`) y siempre escribe en la primera celda libre, sin borrar nada.
| Estado | Función |
|---|---|
| `P_0` | Entra a la candidata en la cinta 1 |
| `P_1` | Si hay símbolo, lo copia; si la cinta 1 lee `λ`, la candidata es **ε** y se escribe solo `#` |
| `P_2` | Copia el resto y escribe `#` al final |
| `P_R` | Rebobina solo la cinta 1; la cinta 3 queda lista para la siguiente impresión |

In [3]:
def construir_delta():
    """Tabla de transiciones. Cada bloque de estados pertenece a una sola responsabilidad:
       G_* generador (cinta 1) | C_* transferencia 1→2 | D_* decisor (cinta 2) | P_* impresora (cinta 3)."""
    delta = {}

    def agregar(q, lee, q_sig, escribe='===', mueve='SSS'):
        # Cada elemento de `lee` es el conjunto de símbolos admisibles en esa cinta; el producto cartesiano
        # expande una regla compacta en transiciones individuales.
        for t in product(*[list(r) for r in lee]):
            esc = tuple(s if e == '=' else e for s, e in zip(t, escribe))
            clave = (q, t)
            # Determinismo: cada (q, a1, a2, a3) tiene a lo sumo una imagen ⇒ δ es función, no relación.
            assert clave not in delta, f"No determinista: {clave}"
            delta[clave] = (q_sig, esc, tuple(mueve))

    b = B
    T2 = '01X' + b        # símbolos que puede haber en la cinta 2
    # Invariante: en G/C/D/P los cabezales 1 y 3 descansan en λ.

    # G: generador shortlex (cinta 1)
    # G_Inicio: la candidata inicial ES ε (cinta 1 vacía); solo cede el control a C.
    agregar('G_Inicio', (b, b, b), 'C_0')
    # G_0..G_R calculan el sucesor shortlex = incrementa en numeral biyectivo base 2 (dígitos {0,1}, 1 = "máximo"):
    #   u0 → u1 ;  u01^k → u10^k ;  1^n → 0^(n+1)   (ε → 0).
    agregar('G_0', (b, b, b), 'G_1', mueve='RSS')                      # entrar a la cadena
    agregar('G_1', ('01', b, b), 'G_1', mueve='RSS')                   # ir al final de la candidata
    agregar('G_1', (b, b, b), 'G_2', mueve='LSS')                      # retroceder al último símbolo
    agregar('G_2', ('1', b, b), 'G_2', escribe='0==', mueve='LSS')     # ACARREO: 1 es el dígito máximo ⇒ vuelve a 0 y propaga a la izquierda
    agregar('G_2', ('0', b, b), 'G_R', escribe='1==', mueve='LSS')     # ABSORCIÓN: 0 → 1 detiene el acarreo; el prefijo u queda intacto
    agregar('G_2', (b, b, b), 'G_E', mueve='RSS')                      # DESBORDAMIENTO: el acarreo llegó a la celda 0
    agregar('G_E', ('0', b, b), 'G_E', mueve='RSS')                    # recorrer 0^n hasta el blanco final
    agregar('G_E', (b, b, b), 'G_R', escribe='0==', mueve='LSS')       # un 0 más: longitud n+1 ⇒ 1^n → 0^(n+1)
    # SINCRONIZACIÓN G→C: cinta 1 rebobinada hasta la celda 0, que es la posición exigida por C_0.
    agregar('G_R', ('01', b, b), 'G_R', mueve='LSS')
    agregar('G_R', (b, b, b), 'C_0')

    # C: transferencia cinta 1 → cinta 2 (copia símbolo a símbolo)
    # Precondición de C_0: ambos cabezales en la celda 0 (la garantizan G_R, D_RWA y D_RWR).
    agregar('C_0', (b, b, b), 'C_1', mueve='RRS')
    for s in '01':
        # Se sobrescribe cada celda de la cinta 2: como |w| nunca decrece en shortlex, la copia cubre
        # todas las X de la ejecución anterior (no hace falta limpiar la cinta 2 aparte).
        agregar('C_1', (s, T2, b), 'C_1', escribe='=' + s + '=', mueve='RRS')
    agregar('C_1', (b, T2, b), 'C_R', mueve='LLS')                    # fin de candidata: rebobinar ambas cintas en paralelo
    # Los dos cabezales recorrieron |w| celdas, así que llegan a la celda 0 en el mismo paso;
    # la lectura (λ, λ) certifica la sincronía antes de ceder el control a D.
    agregar('C_R', ('01', T2, b), 'C_R', mueve='LLS')
    agregar('C_R', (b, b, b), 'D_0')

    # D: decisor de palíndromos (solo trabaja sobre la cinta 2)
    # Recordar su valor en el estado (R0/R1), ir al extremo derecho sin marcar y compararlo; repetir hacia el centro.
    agregar('D_0', (b, b, b), 'D_q0', mueve='SRS')                     # entrar a la cadena
    agregar('D_q0', (b, '0', b), 'D_R0', escribe='=X=', mueve='SRS')   # marca extremo izq. (era 0); el estado R0 lo recuerda
    agregar('D_q0', (b, '1', b), 'D_R1', escribe='=X=', mueve='SRS')   # marca extremo izq. (era 1); el estado R1 lo recuerda
    agregar('D_q0', (b, 'X' + b, b), 'D_Acepta')                       # nada sin marcar (ε o longitud par agotada)
    for k in '01':
        agregar(f'D_R{k}', (b, '01', b), f'D_R{k}', mueve='SRS')       # ir hacia el extremo derecho
        agregar(f'D_R{k}', (b, 'X' + b, b), f'D_L{k}', mueve='SLS')    # pasó el final: retroceder 1
    agregar('D_L0', (b, '0', b), 'D_V', escribe='=X=', mueve='SLS')    # extremos coinciden: marcar y volver
    agregar('D_L0', (b, 'X', b), 'D_Acepta')                           # solo quedaba el centro (longitud impar)
    agregar('D_L0', (b, '1', b), 'D_Rechaza')                          # extremos distintos
    agregar('D_L1', (b, '1', b), 'D_V', escribe='=X=', mueve='SLS')
    agregar('D_L1', (b, 'X', b), 'D_Acepta')
    agregar('D_L1', (b, '0', b), 'D_Rechaza')
    agregar('D_V', (b, '01', b), 'D_V', mueve='SLS')                   # volver al extremo izquierdo sin marcar
    agregar('D_V', (b, 'X', b), 'D_q0', mueve='SRS')

    # REBOBINADO DEL DECISOR: el veredicto se emite con el cabezal 2 a media cadena. Si no se regresa a la
    # celda 0 aquí, el siguiente C_0 (que exige cab2 = 0) arrancaría corrupto en candidatas de longitud ≥ 2.
    # Se emite el veredicto en estados distintos (RWA/RWR) para recordar a qué componente ceder el control.
    agregar('D_Acepta',  (b, 'X' + b, b), 'D_RWA', mueve='SLS')
    agregar('D_Rechaza', (b, '01', b), 'D_RWR', mueve='SLS')
    agregar('D_RWA', (b, '01X', b), 'D_RWA', mueve='SLS')
    agregar('D_RWR', (b, '01X', b), 'D_RWR', mueve='SLS')
    agregar('D_RWA', (b, b, b), 'P_0')                                 # aceptada → impresora
    agregar('D_RWR', (b, b, b), 'G_0')                                 # rechazada → generador; la candidata se descarta

    # P: impresora (copia la cinta 1, que sigue intacta)
    # Se copia desde la cinta 1 (no desde la 2) porque las X del decisor destruyeron la candidata en la cinta 2.
    agregar('P_0', (b, b, b), 'P_1', mueve='RSS')                      # entrar a la candidata (cinta 1)
    for s in '01':
        agregar('P_1', (s, b, b), 'P_2', escribe=f'=={s}', mueve='RSR')    # w ≠ ε: copiar 1er símbolo a la cinta 3

    # CASO ε: |w| = 0 ⇔ la cinta 1 lee λ en la celda 1. Es el único punto donde ε difiere de w ≠ ε:
    # no hay símbolos que copiar y solo se escribe '#'. Así salida = #(w₁#)(w₂#)…, con ε aportando el '#' inicial.
    agregar('P_1', (b, b, b), 'P_R', escribe='==#', mueve='LSR')
    for s in '01':
        agregar('P_2', (s, b, b), 'P_2', escribe=f'=={s}', mueve='RSR')    # copiar el resto
    agregar('P_2', (b, b, b), 'P_R', escribe='==#', mueve='LSR')       # fin de la candidata: escribe "#"

    # REBOBINADO DE LA IMPRESORA: solo la cinta 1 vuelve a la celda 0. La cinta 3 NO se rebobina: su cabezal
    # queda en la 1ª celda libre, lo que garantiza que la salida solo crece y nunca se sobrescribe.
    agregar('P_R', ('01', b, b), 'P_R', mueve='LSS')
    agregar('P_R', (b, b, b), 'G_0')
    return delta

## 4. Validación exhaustiva
- `decidir(w)`: corre la MT completa sobre **una** candidata y devuelve su veredicto.
- `validar(L)` comprueba, sobre todas las cadenas de longitud ≤ `L`:
  1. 19 casos representativos (ε, `0`, `1`, `00`, `01`, `10`, `11`, `010`, `011`, `101`, `111`, `0110`, `0111`, `10101`, `0101` y más).
  2. El generador sigue exactamente el orden shortlex.
  3. El veredicto de la MT coincide con `w == w[::-1]`. Esa comparación es solo de prueba; la MT nunca la usa.
  4. La salida de la cinta 3 es la esperada y solo crece (nunca se sobrescribe).
  5. Los cabezales quedan sincronizados al iniciar cada ciclo.
  6. Las primeras 10 cadenas aceptadas, contando ε, forman `#0#1#00#11#000#010#101#111#0000#`.

**Nota:** con 11 cadenas aceptadas aparece también `0110`; la última celda de esta sección lo muestra.

In [4]:
def decidir(w):
    """Ejecuta la MT  (G→C→D→P) sobre UNA candidata w y devuelve su veredicto."""
    m = MTEnumeradora(construir_delta(), w).ejecutar(n_candidatas=1, max_pasos=10 ** 6)
    return m.veredictos[0][1]


def validar(L=9):
    delta = construir_delta()
    # 1) Casos representativos
    casos = ['', '0', '1', '00', '01', '10', '11', '010', '011', '101', '111',
             '0110', '0111', '10101', '0101', '1001001', '1011010', '110011', '1100110']
    for w in casos:
        assert decidir(w) == ('Acepta' if w == w[::-1] else 'Rechaza'), f"Error en {w!r}"
    print(f"Casos representativos: {len(casos)} correctos")

    # 2) Exhaustiva: orden shortlex + decisión + salida + invariantes de cabezales
    n = 2 ** (L + 1) - 1
    m = MTEnumeradora(delta).ejecutar(n_candidatas=n, guardar_log=False, max_pasos=10 ** 8)
    ref = [''.join(t) for k in range(L + 1) for t in product('01', repeat=k)]
    assert [v[0] for v in m.veredictos] == ref, "El generador no sigue shortlex"
    for w, v, _ in m.veredictos:
        assert v == ('Acepta' if w == w[::-1] else 'Rechaza'), f"Error en {w!r}"
    esperado = '#' + ''.join(w + '#' for w in ref if w and w == w[::-1])
    assert m.salida() == esperado, "La salida de la cinta 3 no coincide"
    # La salida solo crece: cada impresión extiende la anterior
    for a, b_ in zip(m.salidas, m.salidas[1:]):
        assert b_.startswith(a) and len(b_) > len(a)
    # Sincronización: al iniciar cada ciclo, cabezales 1 y 2 en celda 0 y cabezal 3 en la 1ª celda libre
    for c1, c2, c3, ls in m.invariantes:
        assert (c1, c2, c3) == (0, 0, ls + 1), "Cabezal mal reposicionado"
    print(f"Exhaustiva: {len(ref)} candidatas (longitud ≤ {L}) correctas; decisor siempre dentro de su cota.")

    # 3) Primeras 10 cadenas aceptadas (incluyendo ε)
    m10 = MTEnumeradora(delta).ejecutar(n_aceptadas=10, guardar_log=False, max_pasos=10 ** 6)
    assert m10.salida() == '#0#1#00#11#000#010#101#111#0000#'
    print("Primeras 10 aceptadas (incluyendo ε):", m10.salida())

validar()

# Primeras 10 cadenas del lenguaje, incluyendo ε
m = MTEnumeradora(construir_delta()).ejecutar(n_aceptadas=10, guardar_log=False)
print(m.salida())   # #0#1#00#11#000#010#101#111#0000#
# Con 11 cadenas aceptadas (ε + 10 no vacías) aparece también 0110:
m = MTEnumeradora(construir_delta()).ejecutar(n_aceptadas=11, guardar_log=False)
print(m.salida())   # #0#1#00#11#000#010#101#111#0000#0110#

Casos representativos: 19 correctos
Exhaustiva: 1023 candidatas (longitud ≤ 9) correctas; decisor siempre dentro de su cota.
Primeras 10 aceptadas (incluyendo ε): #0#1#00#11#000#010#101#111#0000#
#0#1#00#11#000#010#101#111#0000#
#0#1#00#11#000#010#101#111#0000#0110#


## 5. Visualización

*   Elemento de la lista
*   Elemento de la lista


`TrazadorInteractivo` **no recalcula nada**: lee el log real de la MT (estado, contenido y cabezal de cada cinta en cada paso).

- `render_cinta`: dibuja una cinta como celdas; el cabezal se resalta con `^`.
- `correr(...)`: ejecuta la MT y guarda el log.
- `tabla(idx)`: muestra los últimos 3 pasos hasta `idx`, con el estado y su fase (color por G/C/D/P).
- `mostrar_interfaz()`: **Play**, **slider** de pasos y **menú para saltar** al inicio de cada candidata.

**NOTA**: Si los controles no aparecen en Colab, reinicia el entorno y vuelve a ejecutar todo.

In [5]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

NOMBRES_FASE = {'G': 'Generador', 'C': 'Transferencia 1→2', 'D': 'Decisor', 'P': 'Impresora'}
COLORES_FASE = {'G': '#2980b9', 'C': '#8e44ad', 'D': '#d35400', 'P': '#27ae60'}


def render_cinta(cinta, cab, ancho=22):
    """Dibuja una cinta; ventana de `ancho` celdas alrededor del cabezal si es muy larga."""
    ini = max(0, cab - ancho + 6) if len(cinta) > ancho else 0
    fin = min(len(cinta), ini + ancho)
    h = '<div style="display:flex;gap:3px;font-family:monospace;font-size:14px;padding-bottom:16px;">'
    for i in range(ini, fin):
        if i == cab:
            h += (f'<div style="position:relative;border:2px solid #c0392b;border-radius:4px;padding:3px 7px;'
                  f'font-weight:bold;color:#c0392b;background:#fadbd8;min-width:12px;text-align:center;">{cinta[i]}'
                  f'<span style="position:absolute;bottom:-15px;left:50%;transform:translateX(-50%);font-size:12px;">^</span></div>')
        else:
            h += (f'<div style="border:1px solid #bdc3c7;border-radius:4px;padding:3px 7px;color:#2c3e50;'
                  f'background:#fff;min-width:12px;text-align:center;">{cinta[i]}</div>')
    return h + '</div>'


class TrazadorInteractivo:
    """Visualiza el log REAL de la MT de 3 cintas (no reimplementa ni reconstruye la lógica)."""
    def __init__(self, delta=None):
        self.delta = delta or construir_delta()
        self.m = None

    def correr(self, n_candidatas=None, n_aceptadas=None, candidata_inicial=''):
        self.m = MTEnumeradora(self.delta, candidata_inicial).ejecutar(
            n_candidatas=n_candidatas, n_aceptadas=n_aceptadas, max_pasos=10 ** 6)
        return self.m

    def fila(self, idx, fondo):
        q, cintas, cabs = self.m.log[idx]                  # lectura directa del log
        fase = NOMBRES_FASE.get(q[0], q)
        color = COLORES_FASE.get(q[0], '#2980b9')
        celdas = ''.join(f'<td style="padding:6px 10px;">{render_cinta(c, h)}</td>' for c, h in zip(cintas, cabs))
        return (f'<tr style="background:{fondo};border-bottom:1px solid #eee;">'
                f'<td style="padding:6px;text-align:center;color:#7f8c8d;">{idx}</td>{celdas}'
                f'<td style="padding:6px;text-align:center;font-weight:bold;color:{color};">{q}<br>'
                f'<span style="font-weight:normal;color:#7f8c8d;font-size:11px;">{fase}</span></td></tr>')

    def tabla(self, idx, ventana=3):
        ini = max(0, idx - ventana + 1)
        filas = ''.join(self.fila(i, '#fff3e0' if i == idx else '#f8f9fa') for i in range(ini, idx + 1))
        cand = ''.join(s for s in self.m.log[idx][1][0] if s in '01') or 'ε'
        return (f'<div style="font-family:sans-serif;margin:6px 0;">Candidata en cinta 1: <b>{cand}</b></div>'
                '<table style="width:100%;border-collapse:collapse;font-family:sans-serif;">'
                '<thead><tr style="border-bottom:2px solid #e0e0e0;color:#d35400;"><th>Paso</th>'
                '<th>Cinta 1 (Generador)</th><th>Cinta 2 (Decisor)</th><th>Cinta 3 (Impresora)</th><th>Estado</th></tr></thead>'
                f'<tbody>{filas}</tbody></table>')

    def mostrar_interfaz(self):
        m = self.m
        slider = widgets.IntSlider(0, min=0, max=len(m.log) - 1, step=1, description='Paso:',
                                   layout=widgets.Layout(width='70%'))
        play = widgets.Play(value=0, min=0, max=len(m.log) - 1, step=1, interval=250)
        widgets.jslink((play, 'value'), (slider, 'value'))
        opciones = [(f'#{k+1}  "{c or "ε"}"', i) for k, (i, c) in enumerate(m.inicios)]
        salto = widgets.Dropdown(options=opciones, description='Ir a candidata:', layout=widgets.Layout(width='300px'))
        salto.observe(lambda ch: setattr(slider, 'value', ch['new']), names='value')
        out = widgets.Output()

        def refrescar(*_):
            with out:
                clear_output(wait=True)
                display(HTML(self.tabla(slider.value)))

        slider.observe(refrescar, names='value')
        display(widgets.HBox([play, slider]), salto, out)
        refrescar()

## 6. Demos
### 6a. Traza de una candidata
Cambia `cand` para probar casos: `"0110"` (acepta), `"0111"` (rechaza), una cadena larga o `""` para ε.
Observa que al terminar el decisor la cinta 2 se rebobina antes de pasar a la impresora o al generador.

In [6]:
# Una sola candidata (cambia el texto: "0110" acepta, "0111" rechaza, "1011010" larga, "" = ε)
cand = "0110"
t = TrazadorInteractivo(); t.correr(n_candidatas=1, candidata_inicial=cand); t.mostrar_interfaz()

Dropdown(description='Ir a candidata:', layout=Layout(width='300px'), options=(('#1  "0110"', 0),), value=0)

Output()

### 6b. Ciclo completo del enumerador
Ejecuta generador → transferencia → decisor → impresora hasta imprimir las primeras 10 cadenas del lenguaje (contando ε).
Usa el menú *Ir a candidata* para saltar entre ciclos y observa cómo la cinta 3 solo crece.

In [7]:
# Ciclo completo hasta imprimir las primeras 10 cadenas (incluyendo ε)
t = TrazadorInteractivo(); t.correr(n_aceptadas=10); t.mostrar_interfaz()

Dropdown(description='Ir a candidata:', layout=Layout(width='300px'), options=(('#1  "ε"', 0), ('#2  "0"', 16)…

Output()

## 7. Utilidades para el reporte (PDF)
Estas funciones extraen del código real los datos que el reporte necesita:
- `reporte_base()`: lista de estados por componente, alfabetos Σ y Γ, forma de δ, estado inicial, estados de aceptación/rechazo, orden shortlex y descripción del ciclo.
- `tabla_decisor()` y `mostrar_tabla_decisor()`: tabla de transiciones del decisor.

In [8]:
def _estados_por_fase():
    orden, vistos = [], set()
    for (q, _), (q2, _, _) in construir_delta().items():
        for s in (q, q2):
            if s not in vistos:
                vistos.add(s); orden.append(s)
    return {f: [s for s in orden if s.startswith(f + '_')] for f in 'GCDP'}


def reporte_base():
    delta = construir_delta()
    est = _estados_por_fase()
    S = lambda lst: ', '.join(lst)
    print("A) Estados del generador (G):      ", S(est['G']))
    print("B) Estados de transferencia (C):   ", S(est['C']))
    print("C) Estados del decisor (D):        ", S(est['D']))
    print("D) Estados de la impresora (P):    ", S(est['P']))
    print("E) Alfabeto de entrada Σ:          ", '{' + ', '.join(SIGMA) + '}')
    print("F) Alfabeto de cinta Γ:            ", '{' + ', '.join(GAMMA) + '}', " (λ = blanco, X = marca, # = separador)")
    print(f"G) Función de transición: δ: Q × Γ³ → Q × Γ³ × {{L,S,R}}³ (determinista, 3 cintas); "
          f"{len(delta)} transiciones.")
    print("   Cada transición lee las 3 celdas bajo los cabezales, escribe en las 3 y mueve cada cabezal.")
    print("H) Estado inicial:                  G_Inicio (cinta 1 = ε, cinta 2 vacía, cinta 3 vacía con cabezal en la 1ª celda libre)")
    print("I) Estados de aceptación:           D_Acepta (veredicto del decisor; de aquí pasa a D_RWA → P_*).")
    print("   El enumerador como un todo no se detiene nunca: corre indefinidamente.")
    print("J) Estados de rechazo:              D_Rechaza (veredicto del decisor; de aquí pasa a D_RWR → G_*).")
    print("K) Orden de generación: shortlex (primero por longitud, luego alfabético con 0 < 1):")
    print("   ε, 0, 1, 00, 01, 10, 11, 000, 001, ...  Toda cadena aparece en una posición finita,")
    print("   por lo que todo palíndromo se imprime tras un número finito de pasos.")
    print("L) Ciclo completo:")
    print("   1. G_Inicio/G_*: la cinta 1 contiene la candidata (ε al inicio; luego el sucesor shortlex).")
    print("   2. C_*: se copia la candidata a la cinta 2; ambos cabezales vuelven a la celda 0.")
    print("   3. D_*: el decisor marca extremos con X y los compara hacia el centro; siempre llega a")
    print("      D_Acepta o D_Rechaza porque cada vuelta marca ≥1 símbolo nuevo. Luego rebobina la cinta 2.")
    print("   4. Aceptada → P_*: copia la candidata desde la cinta 1 (intacta) a la 1ª celda libre de la")
    print("      cinta 3 y escribe '#' (para ε escribe solo '#'). Rechazada → se descarta sin imprimir.")
    print("   5. G_*: se calcula la siguiente candidata y se repite desde el paso 2, indefinidamente.")


def tabla_decisor():
    """Filas (estado, lee en cinta 2, escribe, mueve, siguiente) del decisor, agrupando símbolos equivalentes."""
    grupos = {}
    for (q, lect), (q2, esc, mov) in construir_delta().items():
        if not q.startswith('D_'):
            continue
        s, w, m = lect[1], esc[1], mov[1]
        clave = (q, '=' if w == s else w, m, q2)
        grupos.setdefault(clave, []).append(s)
    return [(q, ','.join(ss), w, m, q2) for (q, w, m, q2), ss in grupos.items()]


def mostrar_tabla_decisor():
    filas = tabla_decisor()
    print(f"{'Estado':<10} {'Lee (cinta 2)':<14} {'Escribe':<8} {'Mueve':<6} Siguiente")
    print('-' * 56)
    for q, s, w, m, q2 in filas:
        print(f"{q:<10} {s:<14} {w:<8} {m:<6} {q2}")
    # Versión HTML para copiar al reporte
    h = ('<table style="border-collapse:collapse;font-family:monospace;">'
         '<tr style="background:#eee;"><th>Estado</th><th>Lee</th><th>Escribe</th><th>Mueve</th><th>Siguiente</th></tr>')
    for q, s, w, m, q2 in filas:
        h += ''.join(f'<td style="border:1px solid #ccc;padding:3px 8px;">{x}</td>' for x in (q, s, w, m, q2)).join(('<tr>', '</tr>'))
    display(HTML(h + '</table>'))


reporte_base()
mostrar_tabla_decisor()

A) Estados del generador (G):       G_Inicio, G_0, G_1, G_2, G_R, G_E
B) Estados de transferencia (C):    C_0, C_1, C_R
C) Estados del decisor (D):         D_0, D_q0, D_R0, D_R1, D_Acepta, D_L0, D_L1, D_V, D_Rechaza, D_RWA, D_RWR
D) Estados de la impresora (P):     P_0, P_1, P_2, P_R
E) Alfabeto de entrada Σ:           {0, 1}
F) Alfabeto de cinta Γ:             {λ, 0, 1, X, #}  (λ = blanco, X = marca, # = separador)
G) Función de transición: δ: Q × Γ³ → Q × Γ³ × {L,S,R}³ (determinista, 3 cintas); 79 transiciones.
   Cada transición lee las 3 celdas bajo los cabezales, escribe en las 3 y mueve cada cabezal.
H) Estado inicial:                  G_Inicio (cinta 1 = ε, cinta 2 vacía, cinta 3 vacía con cabezal en la 1ª celda libre)
I) Estados de aceptación:           D_Acepta (veredicto del decisor; de aquí pasa a D_RWA → P_*).
   El enumerador como un todo no se detiene nunca: corre indefinidamente.
J) Estados de rechazo:              D_Rechaza (veredicto del decisor; de aquí pasa a D_RWR →

Estado,Lee,Escribe,Mueve,Siguiente
D_0,λ,=,R,D_q0
D_q0,0,X,R,D_R0
D_q0,1,X,R,D_R1
D_q0,"X,λ",=,S,D_Acepta
D_R0,"0,1",=,R,D_R0
D_R0,"X,λ",=,L,D_L0
D_R1,"0,1",=,R,D_R1
D_R1,"X,λ",=,L,D_L1
D_L0,0,X,L,D_V
D_L0,X,=,S,D_Acepta
